# AlpCAN CT Pipeline: Nodül Karakterizasyon Eğitimi

**Amaç:** LIDC-IDRI veri seti üzerinde nodül karakterizasyon modeli eğitmek.  
AlpCAN CT Pipeline Agent 3 (Nodül Karakterizasyon) için ağırlık üretimi.

**İçindekiler:**
1. LIDC-IDRI veri seti yükleme ve nodül patch çıkarımı
2. Etiket hazırlama: Boyut bazlı sınıflandırma (Binary + 4-sınıf risk)
3. Etiket dağılımı görselleştirme
4. Eğitim/doğrulama bölmesi + Dataset sınıfı
5. ResNet-50 + CBAM modeli (Channel & Spatial Attention)
6. Focal Loss + Weighted CE + AdamW + CosineAnnealing
7. Eğitim döngüsü (40 epoch, AMP, multi-task loss)
8. Eğitim grafikleri (loss, AUC, risk_acc, LR)
9. ROC eğrisi + confusion matrix
10. Grad-CAM görselleştirme
11. Lung-RADS eşleme
12. Sonuçları kaydetme + özet

**Model:** ResNet-50 + CBAM (Channel & Spatial Attention)  
**GPU:** Kaggle T4 16GB  
**Dataset:** `zhangweiled/lidcidri` — LIDC-IDRI kesilmiş nodül dilimleri

**Not:** LIDC-IDRI kesilmiş dilim formatı 2D'dir. Tam 3D karakterizasyon gelecek fazda  
tam volümetrik BT serileri ile yapılacaktır. Bu notebook 2D slice-level baseline oluşturur.

In [ ]:
!pip install -q grad-cam albumentations

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. LIDC-IDRI Veri Seti ve Nodül Patch Çıkarımı

In [ ]:
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LIDC-IDRI dataset
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")
if not DATA_DIR.exists():
    for candidate in INPUT_DIR.rglob("LIDC-IDRI-0001"):
        DATA_DIR = candidate.parent
        break
    if not DATA_DIR.exists():
        raise FileNotFoundError("LIDC-IDRI dataset bulunamadi!")

print(f"Dataset: {DATA_DIR}")

patient_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Toplam hasta: {len(patient_dirs)}")

In [ ]:
# Her nodul icin: orta dilim + maske boyutu + annotator sayisi
# Malignite etiketi: maskelerin varligini/boyutunu proxy olarak kullan
# (Gercek LIDC-IDRI XML annotasyonlari bu dataset'te yok,
#  maske boyutu ve annotator uyumu ile malignite tahmini yapilir)

nodule_data = []

for pdir in patient_dirs:
    nodule_dirs = sorted([d for d in pdir.iterdir() if d.is_dir()])
    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs = sorted([d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")])

        if not img_dir.exists() or not mask_dirs:
            continue

        img_files = sorted(img_dir.glob("*.png"))
        if not img_files:
            continue

        # Orta dilimi sec (nodul en buyuk oldugu dilim)
        mid_idx = len(img_files) // 2
        mid_img = img_files[mid_idx]

        # Tum annotator maskelerinden boyut hesapla
        mask_areas = []
        mask_agreement = 0
        combined_mask = None

        for mdir in mask_dirs:
            mask_file = mdir / mid_img.name
            if mask_file.exists():
                m = np.array(Image.open(mask_file).convert('L'))
                area = (m > 0).sum()
                mask_areas.append(area)
                if combined_mask is None:
                    combined_mask = (m > 0).astype(np.float32)
                else:
                    combined_mask += (m > 0).astype(np.float32)

        if not mask_areas or combined_mask is None:
            continue

        avg_area = np.mean(mask_areas)
        n_annotators = len(mask_areas)
        consensus_area = ((combined_mask / n_annotators) >= 0.5).sum()

        # Nodul capini piksel olarak hesapla
        diameter_px = np.sqrt(consensus_area / np.pi) * 2 if consensus_area > 0 else 0

        nodule_data.append({
            'patient_id': pdir.name,
            'nodule': ndir.name,
            'image_path': str(mid_img),
            'mask_dirs': [str(m) for m in mask_dirs],
            'n_annotators': n_annotators,
            'avg_mask_area': avg_area,
            'consensus_area': float(consensus_area),
            'diameter_px': float(diameter_px),
            'n_slices': len(img_files),
        })

df = pd.DataFrame(nodule_data)
print(f"\nToplam nodul: {len(df)}")
print(f"Toplam hasta: {df['patient_id'].nunique()}")
print(f"\nNodul cap (piksel) istatistikleri:")
print(df['diameter_px'].describe())

---
## 2. Etiket Hazırlama — Boyut Bazlı Sınıflandırma

In [ ]:
# Lung-RADS benzeri boyut siniflandirmasi (piksel bazli proxy)
# Gercek mm olcusu icin DICOM spacing bilgisi gerekli
# Bu dataset'te spacing bilgisi yok, piksel boyutunu kullaniyoruz

median_diameter = df['diameter_px'].median()
print(f"Medyan nodul capi: {median_diameter:.1f} piksel")

# Boyut kategorizasyonu
df['size_category'] = pd.cut(
    df['diameter_px'],
    bins=[0, median_diameter * 0.5, median_diameter, median_diameter * 2, float('inf')],
    labels=['cok_kucuk', 'kucuk', 'orta', 'buyuk']
)

# Binary: Suspicious etiketi -> buyuk nodul VE yuksek annotator uyumu
df['suspicious'] = ((df['diameter_px'] > median_diameter) & (df['n_annotators'] >= 3)).astype(int)

# Multi-class: 4 sinif risk
def assign_risk_class(row):
    """Nodul capina gore risk sinifi ata.
    
    0: Lung-RADS 1 benzeri (Dusuk Risk)
    1: Lung-RADS 2 benzeri (Orta-Dusuk Risk)
    2: Lung-RADS 3 benzeri (Orta-Yuksek Risk)
    3: Lung-RADS 4A benzeri (Yuksek Risk)
    """
    d = row['diameter_px']
    if d <= median_diameter * 0.5:
        return 0  # Lung-RADS 1 benzeri
    elif d <= median_diameter:
        return 1  # Lung-RADS 2 benzeri
    elif d <= median_diameter * 2:
        return 2  # Lung-RADS 3 benzeri
    else:
        return 3  # Lung-RADS 4 benzeri

df['risk_class'] = df.apply(assign_risk_class, axis=1)

print(f"\nBinary siniflandirma (suspicious):")
print(df['suspicious'].value_counts())
print(f"\nRisk sinifi dagilimi:")
print(df['risk_class'].value_counts().sort_index())
print(f"\nBoyut kategorisi:")
print(df['size_category'].value_counts())

---
## 3. Etiket Dağılımı Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Nodul cap dagilimi
axes[0].hist(df['diameter_px'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=median_diameter, color='red', linestyle='--', label=f'Medyan: {median_diameter:.1f}')
axes[0].set_title('Nodul Cap Dagilimi', fontweight='bold')
axes[0].set_xlabel('Cap (piksel)')
axes[0].set_ylabel('Sayi')
axes[0].legend()

# Annotator uyumu
ann_counts = df['n_annotators'].value_counts().sort_index()
axes[1].bar(ann_counts.index, ann_counts.values, color='coral', edgecolor='black')
axes[1].set_title('Annotator Uyumu', fontweight='bold')
axes[1].set_xlabel('Annotator Sayisi')
axes[1].set_ylabel('Nodul Sayisi')

# Risk sinifi
risk_counts = df['risk_class'].value_counts().sort_index()
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
risk_labels = ['Dusuk Risk', 'Orta-Dusuk', 'Orta-Yuksek', 'Yuksek Risk']
axes[2].bar(risk_labels, [risk_counts.get(i, 0) for i in range(4)],
            color=colors, edgecolor='black')
axes[2].set_title('Risk Sinifi Dagilimi', fontweight='bold')
axes[2].set_ylabel('Nodul Sayisi')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Eğitim/Doğrulama Bölmesi ve Dataset

In [ ]:
# Hasta bazli bolme (veri sizintisi onleme)
np.random.seed(42)
patients = df['patient_id'].unique()
np.random.shuffle(patients)

split_idx = int(len(patients) * 0.8)
train_patients = set(patients[:split_idx])
val_patients = set(patients[split_idx:])

train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)

print(f"Train: {len(train_df)} nodul ({len(train_patients)} hasta)")
print(f"Val:   {len(val_df)} nodul ({len(val_patients)} hasta)")
print(f"\nTrain suspicious orani: {train_df['suspicious'].mean():.2%}")
print(f"Val suspicious orani:   {val_df['suspicious'].mean():.2%}")

In [ ]:
PATCH_SIZE = 128

train_transform = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


class NoduleCharDataset(Dataset):
    """Nodul karakterizasyon dataset'i.

    Her nodul icin orta dilim goruntusunu yukler.
    Maskeleri kullanarak nodul bolgesine crop yapar ve context ekler.
    """
    def __init__(self, dataframe, transform=None, crop_to_nodule=True):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.crop_to_nodule = crop_to_nodule

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Goruntu oku
        img = np.array(Image.open(row['image_path']).convert('L'))

        if self.crop_to_nodule and row['mask_dirs']:
            # Konsensus maske ile nodul bolgesini bul
            combined = np.zeros(img.shape, dtype=np.float32)
            n = 0
            for mdir in row['mask_dirs']:
                mask_file = Path(mdir) / Path(row['image_path']).name
                if mask_file.exists():
                    m = np.array(Image.open(mask_file).convert('L'))
                    # Boyut uyumsuzlugunu kontrol et
                    if m.shape == img.shape:
                        combined += (m > 0).astype(np.float32)
                        n += 1

            if n > 0:
                consensus = (combined / n) >= 0.5
                if consensus.any():
                    # Nodul merkezi ve genisletilmis crop
                    rows_with = np.where(np.any(consensus, axis=1))[0]
                    cols_with = np.where(np.any(consensus, axis=0))[0]

                    cy = (rows_with[0] + rows_with[-1]) // 2
                    cx = (cols_with[0] + cols_with[-1]) // 2
                    h = rows_with[-1] - rows_with[0]
                    w = cols_with[-1] - cols_with[0]

                    # 2x nodul boyutu kadar context ekle
                    margin = max(h, w)
                    half = max(margin, 32)  # minimum 32 piksel

                    y1 = max(0, cy - half)
                    y2 = min(img.shape[0], cy + half)
                    x1 = max(0, cx - half)
                    x2 = min(img.shape[1], cx + half)

                    img = img[y1:y2, x1:x2]

        # Bos veya cok kucuk goruntu kontrolu
        if img.size == 0 or img.shape[0] < 4 or img.shape[1] < 4:
            img = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)

        # 3 kanala cevir (ImageNet pretrained ResNet icin)
        img_rgb = np.stack([img, img, img], axis=-1)

        if self.transform:
            augmented = self.transform(image=img_rgb)
            img_tensor = augmented['image']
        else:
            img_tensor = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 255.0

        # Etiketler
        suspicious = torch.tensor(row['suspicious'], dtype=torch.float32)
        risk_class = torch.tensor(row['risk_class'], dtype=torch.long)

        return img_tensor, suspicious, risk_class


train_dataset = NoduleCharDataset(train_df, transform=train_transform)
val_dataset = NoduleCharDataset(val_df, transform=val_transform)

BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"Patch boyutu: {PATCH_SIZE}x{PATCH_SIZE}")

img_sample, susp, risk = train_dataset[0]
print(f"Goruntu shape: {img_sample.shape}, Suspicious: {susp.item()}, Risk: {risk.item()}")

---
## 5. Model — ResNet-50 + CBAM

In [ ]:
class ChannelAttention(nn.Module):
    """CBAM Channel Attention.
    
    Kanal bazinda onem agirliklarini ogrenme.
    Global Average Pool + Global Max Pool -> shared MLP -> sigmoid.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        attention = torch.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * attention


class SpatialAttention(nn.Module):
    """CBAM Spatial Attention.
    
    Mekansal dikkat haritasi: channel-wise avg + max -> conv -> sigmoid.
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        attention = torch.sigmoid(self.conv(concat))
        return x * attention


class CBAM(nn.Module):
    """Convolutional Block Attention Module.
    
    Ref: Woo et al. 'CBAM: Convolutional Block Attention Module' (ECCV 2018)
    """
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x


class ResNet50CBAM(nn.Module):
    """ResNet-50 + CBAM nodul karakterizasyon modeli.

    Cikislar:
    - suspicious: Binary (0/1) -- nodul supheli mi?
    - risk_class: 4-sinif (0-3) -- risk seviyesi
    """
    def __init__(self, n_risk_classes=4):
        super().__init__()
        # ResNet-50 backbone (ImageNet pre-trained)
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        # Son FC ve avgpool haric tum katmanlar
        self.features = nn.Sequential(*list(resnet.children())[:-2])

        # CBAM attention (2048 kanal -- ResNet-50 son layer cikisi)
        self.cbam = CBAM(2048, reduction=16)

        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Classification heads
        self.dropout = nn.Dropout(0.3)
        self.fc_suspicious = nn.Linear(2048, 1)  # Binary
        self.fc_risk = nn.Linear(2048, n_risk_classes)  # Multi-class

    def forward(self, x):
        features = self.features(x)        # (B, 2048, H/32, W/32)
        features = self.cbam(features)      # CBAM attention uygula
        pooled = self.gap(features).flatten(1)  # (B, 2048)
        pooled = self.dropout(pooled)

        suspicious = self.fc_suspicious(pooled).squeeze(-1)  # (B,)
        risk = self.fc_risk(pooled)                          # (B, 4)

        return suspicious, risk, features  # features for Grad-CAM


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet50CBAM(n_risk_classes=4).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: ResNet-50 + CBAM")
print(f"Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"Device: {device}")
print(f"Cikislar: suspicious (binary), risk_class (4-sinif)")

---
## 6. Loss ve Optimizer

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss -- class imbalance icin.
    
    Kolay orneklerin loss katkisini azaltir, zor orneklere odaklanir.
    Ref: Lin et al. 'Focal Loss for Dense Object Detection' (ICCV 2017)
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()


# Class weights hesapla (risk sinifi imbalance icin)
risk_counts = train_df['risk_class'].value_counts().sort_index()
risk_weights = 1.0 / (risk_counts.values.astype(np.float32) + 1)
risk_weights = risk_weights / risk_weights.sum() * len(risk_weights)
risk_weights_tensor = torch.tensor(risk_weights, dtype=torch.float32).to(device)
print(f"Risk class weights: {risk_weights}")

criterion_suspicious = FocalLoss(alpha=0.25, gamma=2.0)
criterion_risk = nn.CrossEntropyLoss(weight=risk_weights_tensor)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-6)

print("Suspicious loss: Focal (alpha=0.25, gamma=2)")
print("Risk loss: Weighted CrossEntropy")
print("Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print("Scheduler: CosineAnnealingLR (T_max=40)")

---
## 7. Eğitim

In [ ]:
NUM_EPOCHS = 80
best_val_auc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_risk_acc': [], 'lr': []}
scaler = torch.cuda.amp.GradScaler()

SUSPICIOUS_WEIGHT = 1.0
RISK_WEIGHT = 0.5

print(f"Egitim basliyor -- {NUM_EPOCHS} epoch")
print(f"Loss agirliklari: suspicious={SUSPICIOUS_WEIGHT}, risk={RISK_WEIGHT}")
print("=" * 70)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    n_train = 0

    for images, susp_labels, risk_labels in train_loader:
        images = images.to(device)
        susp_labels = susp_labels.to(device)
        risk_labels = risk_labels.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            susp_pred, risk_pred, _ = model(images)
            loss_susp = criterion_suspicious(susp_pred, susp_labels)
            loss_risk = criterion_risk(risk_pred, risk_labels)
            loss = SUSPICIOUS_WEIGHT * loss_susp + RISK_WEIGHT * loss_risk

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        n_train += images.size(0)

    train_loss /= max(n_train, 1)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    all_susp_preds = []
    all_susp_labels = []
    all_risk_preds = []
    all_risk_labels = []
    n_val = 0

    with torch.no_grad():
        for images, susp_labels, risk_labels in val_loader:
            images = images.to(device)
            susp_labels = susp_labels.to(device)
            risk_labels = risk_labels.to(device)

            with torch.cuda.amp.autocast():
                susp_pred, risk_pred, _ = model(images)
                loss_susp = criterion_suspicious(susp_pred, susp_labels)
                loss_risk = criterion_risk(risk_pred, risk_labels)
                loss = SUSPICIOUS_WEIGHT * loss_susp + RISK_WEIGHT * loss_risk

            val_loss += loss.item() * images.size(0)
            all_susp_preds.extend(torch.sigmoid(susp_pred).cpu().numpy())
            all_susp_labels.extend(susp_labels.cpu().numpy())
            all_risk_preds.extend(risk_pred.argmax(dim=1).cpu().numpy())
            all_risk_labels.extend(risk_labels.cpu().numpy())
            n_val += images.size(0)

    val_loss /= max(n_val, 1)

    # Metrikler
    all_susp_preds = np.array(all_susp_preds)
    all_susp_labels = np.array(all_susp_labels)
    all_risk_preds = np.array(all_risk_preds)
    all_risk_labels = np.array(all_risk_labels)

    if len(np.unique(all_susp_labels)) > 1:
        val_auc = roc_auc_score(all_susp_labels, all_susp_preds)
    else:
        val_auc = 0.0

    risk_acc = (all_risk_preds == all_risk_labels).mean()

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    history['val_risk_acc'].append(risk_acc)
    history['lr'].append(current_lr)

    # Best model kaydet
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_auc': best_val_auc,
            'val_risk_acc': risk_acc,
        }, OUTPUT_DIR / 'ct_char_best_model.pth')
        marker = ' << BEST'
    else:
        marker = ''

    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1:>3d}/{NUM_EPOCHS} | "
              f"Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | "
              f"AUC: {val_auc:.4f} | "
              f"Risk Acc: {risk_acc:.4f} | "
              f"LR: {current_lr:.6f} | "
              f"{elapsed/60:.1f}m{marker}")

total_time = time.time() - start_time
print(f"\nEgitim tamamlandi! Sure: {total_time/60:.1f} dk")
print(f"En iyi AUC: {best_val_auc:.4f}")


---
## 8. Eğitim Grafikleri

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0, 0].set_title('Loss', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# AUC
axes[0, 1].plot(history['val_auc'], linewidth=2, color='#e74c3c')
axes[0, 1].set_title('Suspicious AUC', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('AUC')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0, 1)

# Risk Accuracy
axes[1, 0].plot(history['val_risk_acc'], linewidth=2, color='#3498db')
axes[1, 0].set_title('Risk Classification Accuracy', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim(0, 1)

# Learning Rate
axes[1, 1].plot(history['lr'], linewidth=2, color='#2ecc71')
axes[1, 1].set_title('Learning Rate (Cosine)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('LR')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. ROC Eğrisi + Confusion Matrix

In [ ]:
# En iyi modeli yukle ve detayli degerlendirme
checkpoint = torch.load(OUTPUT_DIR / 'ct_char_best_model.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Best model yuklendi (Epoch {checkpoint['epoch']+1}, AUC={checkpoint['val_auc']:.4f})")

# Tam validasyon tahminleri
all_susp_preds = []
all_susp_labels = []
all_risk_preds = []
all_risk_labels = []
all_risk_probs = []

with torch.no_grad():
    for images, susp_labels, risk_labels in val_loader:
        images = images.to(device)
        with torch.cuda.amp.autocast():
            susp_pred, risk_pred, _ = model(images)
        all_susp_preds.extend(torch.sigmoid(susp_pred).cpu().numpy())
        all_susp_labels.extend(susp_labels.numpy())
        all_risk_preds.extend(risk_pred.argmax(dim=1).cpu().numpy())
        all_risk_labels.extend(risk_labels.numpy())
        all_risk_probs.extend(F.softmax(risk_pred, dim=1).cpu().numpy())

all_susp_preds = np.array(all_susp_preds)
all_susp_labels = np.array(all_susp_labels)
all_risk_preds = np.array(all_risk_preds)
all_risk_labels = np.array(all_risk_labels)
all_risk_probs = np.array(all_risk_probs)

# --- Gorsellestirme ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC egrisi
if len(np.unique(all_susp_labels)) > 1:
    fpr, tpr, thresholds = roc_curve(all_susp_labels, all_susp_preds)
    auc_val = roc_auc_score(all_susp_labels, all_susp_preds)

    axes[0].plot(fpr, tpr, linewidth=2, color='#e74c3c', label=f'AUC = {auc_val:.4f}')
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[0].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Egrisi -- Suspicious Detection', fontweight='bold')
    axes[0].legend(fontsize=12)
    axes[0].grid(True, alpha=0.2)
else:
    auc_val = 0.0
    axes[0].text(0.5, 0.5, 'Tek sinif -- ROC hesaplanamadi',
                ha='center', va='center', transform=axes[0].transAxes, fontsize=12)
    axes[0].set_title('ROC Egrisi', fontweight='bold')

# Confusion matrix
cm = confusion_matrix(all_risk_labels, all_risk_preds)
risk_names = ['Dusuk', 'Orta-D', 'Orta-Y', 'Yuksek']

# Sadece mevcut siniflari goster
present_classes = sorted(set(all_risk_labels) | set(all_risk_preds))
present_names = [risk_names[i] for i in present_classes if i < len(risk_names)]

im = axes[1].imshow(cm, interpolation='nearest', cmap='Blues')
axes[1].set_title('Risk Sinifi Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Tahmin')
axes[1].set_ylabel('Gercek')
axes[1].set_xticks(range(cm.shape[1]))
axes[1].set_yticks(range(cm.shape[0]))
axes[1].set_xticklabels(present_names[:cm.shape[1]])
axes[1].set_yticklabels(present_names[:cm.shape[0]])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black',
                    fontsize=12, fontweight='bold')

plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRisk Sinifi Classification Report:")
print(classification_report(
    all_risk_labels, all_risk_preds,
    target_names=present_names[:len(set(all_risk_labels))],
    zero_division=0
))

---
## 10. Grad-CAM Görselleştirme

In [ ]:
class GradCAM:
    """Basit Grad-CAM implementasyonu.
    
    Modelin dikkat ettigi bolgeleri gorsellestirir.
    register_full_backward_hook kullanir (deprecated register_backward_hook yerine).
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._handles = []

        # Hook'lari kaydet (temizlik icin handle'lari sakla)
        h1 = target_layer.register_forward_hook(self._forward_hook)
        h2 = target_layer.register_full_backward_hook(self._backward_hook)
        self._handles.extend([h1, h2])

    def _forward_hook(self, module, input, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        """Grad-CAM haritasi uret.
        
        Args:
            input_tensor: (1, C, H, W) girdi tensoru
            target_class: Hedef sinif indeksi (None = suspicious head)
            
        Returns:
            cam: (H, W) normalize edilmis dikkat haritasi [0, 1]
        """
        self.model.eval()

        # Forward pass (gradient hesaplamasi icin enable)
        input_tensor = input_tensor.detach().requires_grad_(False)
        # Model parametrelerinde grad gerekli, input'ta degil
        for p in self.model.parameters():
            p.requires_grad_(True)

        output, risk_out, _ = self.model(input_tensor)

        if target_class is None:
            target = output
        else:
            target = output[:, target_class] if output.dim() > 1 else output

        self.model.zero_grad()
        target.sum().backward(retain_graph=False)

        if self.gradients is None or self.activations is None:
            # Fallback: sifir harita dondur
            h, w = input_tensor.shape[2:]
            return np.zeros((h, w), dtype=np.float32)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam - cam.min()
        cam_max = cam.max()
        if cam_max > 0:
            cam = cam / cam_max
        return cam.cpu().numpy()[0, 0]

    def remove_hooks(self):
        """Hook'lari temizle."""
        for h in self._handles:
            h.remove()
        self._handles.clear()


def find_target_layer(model):
    """ResNet-50 icin Grad-CAM hedef katmanini guvenli sekilde bul.
    
    Son bottleneck blogunun son convolutional katmanini arar.
    Bulamazsa features'in son katmanini dondurur.
    """
    # ResNet-50 features[-1] = layer4, layer4[-1] = son Bottleneck
    try:
        last_block = model.features[-1][-1]
        # Bottleneck'te conv3 (1x1 conv) var
        if hasattr(last_block, 'conv3'):
            return last_block.conv3
        # conv2 (3x3 conv) de iyi bir hedef
        elif hasattr(last_block, 'conv2'):
            return last_block.conv2
        else:
            return last_block
    except (IndexError, AttributeError):
        # Fallback: features'in kendisi
        return model.features[-1]


# Hedef katmani bul ve Grad-CAM olustur
target_layer = find_target_layer(model)
print(f"Grad-CAM hedef katman: {target_layer.__class__.__name__}")
grad_cam = GradCAM(model, target_layer)

# 8 ornek icin Grad-CAM gorsellestir
n_samples = min(8, len(val_dataset))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Grad-CAM -- Model Dikkat Haritasi', fontsize=14, fontweight='bold')

indices = np.linspace(0, len(val_dataset) - 1, n_samples, dtype=int)

for col_idx, (ax, idx) in enumerate(zip(axes.flat, indices)):
    img, susp_label, risk_label = val_dataset[idx]
    img_gpu = img.unsqueeze(0).to(device)

    cam = grad_cam.generate(img_gpu)

    # Goruntu denormalize
    img_np = img.numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = (img_np * std + mean).clip(0, 1)

    ax.imshow(img_np[:, :, 0], cmap='gray')
    ax.imshow(cam, cmap='jet', alpha=0.4)

    with torch.no_grad():
        susp_pred, risk_pred, _ = model(img_gpu)
        susp_score = torch.sigmoid(susp_pred).item()
        risk_class_pred = risk_pred.argmax(dim=1).item()

    risk_names_short = ['DR', 'OD', 'OY', 'YR']
    true_risk = risk_names_short[risk_label] if risk_label < len(risk_names_short) else '?'
    pred_risk = risk_names_short[risk_class_pred] if risk_class_pred < len(risk_names_short) else '?'
    ax.set_title(f'S:{susp_score:.2f} | GT:{true_risk} P:{pred_risk}', fontsize=9)
    ax.axis('off')

# Kalan bos eksenleri gizle (8'den az ornek varsa)
for ax_idx in range(n_samples, len(axes.flat)):
    axes.flat[ax_idx].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_char_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()

# Hook'lari temizle
grad_cam.remove_hooks()

---
## 11. Lung-RADS Eşleme ve Sonuçları Kaydetme

In [ ]:
# Risk sinifini Lung-RADS kategorisine esle
RISK_TO_LUNGRADS = {
    0: '1',    # Dusuk risk -> Lung-RADS 1 (Negatif)
    1: '2',    # Orta-Dusuk -> Lung-RADS 2 (Benign)
    2: '3',    # Orta-Yuksek -> Lung-RADS 3 (Muhtemelen Benign)
    3: '4A',   # Yuksek risk -> Lung-RADS 4A (Suphe)
}

LUNGRADS_DESCRIPTIONS = {
    '1':  'Negatif -- Nodul yok veya kesinlikle benign',
    '2':  'Benign -- Benign gorunum veya davranisin nodul',
    '3':  'Muhtemelen Benign -- Muhtemelen benign, kisa sureli takip',
    '4A': 'Supheli -- Ek tani veya doku orneklemesi gerekebilir',
}

print("Risk Sinifi -> Lung-RADS Esleme:")
print("-" * 65)
for risk, lr in RISK_TO_LUNGRADS.items():
    n_gt = int((all_risk_labels == risk).sum())
    n_pred = int((all_risk_preds == risk).sum())
    desc = LUNGRADS_DESCRIPTIONS[lr]
    print(f"  Risk {risk} -> Lung-RADS {lr}: {n_gt:>4d} gercek, {n_pred:>4d} tahmin")
    print(f"           {desc}")

# Sonuclari kaydet
results_df = val_df.copy()
results_df['susp_pred'] = all_susp_preds
results_df['risk_pred'] = all_risk_preds
results_df['lung_rads_pred'] = [RISK_TO_LUNGRADS.get(int(r), '?') for r in all_risk_preds]

# Risk class olasilik kolonlari ekle
for i, name in enumerate(['risk_prob_low', 'risk_prob_mid_low', 'risk_prob_mid_high', 'risk_prob_high']):
    if i < all_risk_probs.shape[1]:
        results_df[name] = all_risk_probs[:, i]

# mask_dirs kolonu liste icerir, CSV'ye string olarak kaydet
results_df['mask_dirs'] = results_df['mask_dirs'].apply(str)
results_df.to_csv(OUTPUT_DIR / 'ct_char_predictions.csv', index=False)
print(f"\nTahminler kaydedildi: ct_char_predictions.csv")

# Metrikler
final_auc = roc_auc_score(all_susp_labels, all_susp_preds) if len(np.unique(all_susp_labels)) > 1 else 0.0
final_risk_acc = float((all_risk_preds == all_risk_labels).mean())

metrics = {
    'suspicious_auc': final_auc,
    'risk_accuracy': final_risk_acc,
    'n_val_samples': len(all_susp_labels),
    'n_train_samples': len(train_df),
    'best_epoch': int(checkpoint['epoch'] + 1),
    'total_epochs': NUM_EPOCHS,
}
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'ct_char_metrics.csv', index=False)
print(f"Metrikler kaydedildi: ct_char_metrics.csv")

# Training history
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'ct_char_training_history.csv', index=False)
print(f"Egitim historisi kaydedildi: ct_char_training_history.csv")

print(f"\nKaydedilen dosyalar:")
for f in sorted(OUTPUT_DIR.glob('ct_char_*')):
    size_kb = f.stat().st_size / 1024
    if size_kb > 1024:
        print(f"  {f.name} ({size_kb/1024:.1f} MB)")
    else:
        print(f"  {f.name} ({size_kb:.0f} KB)")

In [ ]:
print("\n" + "=" * 65)
print("AlpCAN CT Pipeline -- Nodul Karakterizasyon Egitimi OZET")
print("=" * 65)

print(f"\n--- Dataset ---")
print(f"  LIDC-IDRI (zhangweiled/lidcidri)")
print(f"  Train: {len(train_df)} nodul ({len(train_patients)} hasta)")
print(f"  Val:   {len(val_df)} nodul ({len(val_patients)} hasta)")
print(f"  Etiket: Boyut bazli risk siniflandirma (4 sinif)")
print(f"  Suspicious: cap > medyan VE n_annotators >= 3")

print(f"\n--- Model ---")
print(f"  Mimari: ResNet-50 + CBAM (Channel & Spatial Attention)")
print(f"  Giris: {PATCH_SIZE}x{PATCH_SIZE}x3 (nodul patch, context dahil)")
print(f"  Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Loss: Focal (suspicious) + Weighted CE (risk)")
print(f"  Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print(f"  Scheduler: CosineAnnealingLR (T_max={NUM_EPOCHS})")
print(f"  Epoch: {NUM_EPOCHS}")
print(f"  AMP: Evet (torch.cuda.amp)")

print(f"\n--- Performans ---")
print(f"  Suspicious AUC: {metrics['suspicious_auc']:.4f}")
print(f"  Risk Accuracy:  {metrics['risk_accuracy']:.4f}")
print(f"  Best Epoch:     {metrics['best_epoch']}")

print(f"\n--- Lung-RADS Esleme ---")
for risk, lr in RISK_TO_LUNGRADS.items():
    print(f"  Risk {risk} -> Lung-RADS {lr}: {LUNGRADS_DESCRIPTIONS[lr]}")

print(f"\n--- Cikti Dosyalari ---")
print(f"  ct_char_best_model.pth       -- En iyi model agirliklari")
print(f"  ct_char_metrics.csv           -- Final metrikler")
print(f"  ct_char_training_history.csv  -- Epoch bazli egitim historisi")
print(f"  ct_char_training_curves.png   -- Egitim grafikleri")
print(f"  ct_char_evaluation.png        -- ROC + Confusion Matrix")
print(f"  ct_char_gradcam.png           -- Grad-CAM dikkat haritalari")
print(f"  ct_char_predictions.csv       -- Validasyon tahminleri")
print(f"  ct_char_label_distribution.png-- Etiket dagilim grafigi")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Model agirliklarini AlpCAN backend'e entegre et")
print(f"  2. characterization_inference.py guncelle")
print(f"  3. Nodul segmentasyon + karakterizasyon pipeline birlestir")
print(f"  4. Gercek LIDC-IDRI XML annotasyonlarindan malignite etiketleri")
print(f"  5. 3D volumetrik analiz (tam BT serisi)")
print("=" * 65)